Old version

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from pandas_datareader import data as pdr


START = "1995-01-01"
END   = "2025-12-01"
MARKET = "SPY"

FRED_SERIES = {
    "DFF": "fed_funds",
    "DGS10": "rate_10y",
    "DGS2": "rate_2y",
    "T10Y2Y": "yc_slope_10y2y",
    "CPIAUCSL": "cpi",
    "UNRATE": "unemp",
    "INDPRO": "indpro",
    "VIXCLS": "vix",
    "DBAA": "dbaa",
    "DAAA": "daaa",
}

SPLITS = {
    "gfc_2008_2010": ("2008-01-01", "2010-12-31"),
    "covid_2020_2021": ("2020-01-01", "2021-12-31"),
    "inflation_2022_2023": ("2022-01-01", "2023-12-31"),
}

OUT_ROOT = "dataset_finance_fred2"


def ensure_dirs():
    os.makedirs(os.path.join(OUT_ROOT, "eval_jsonl_no_fred"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "metadata"), exist_ok=True)


def get_sp500_tickers() -> list[str]:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    tickers = pd.read_html(response.text)[0]["Symbol"].astype(str).tolist()
    return [t.replace(".", "-") for t in tickers]


def download_close(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    px = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    price_field = "Close"

    if isinstance(px.columns, pd.MultiIndex):
        close = pd.concat(
            {t: px[t][price_field] for t in tickers if t in px.columns.get_level_values(0)},
            axis=1,
        )
    else:
        close = px[price_field].to_frame(tickers[0])

    return close.sort_index()


def log_returns(close_df: pd.DataFrame) -> pd.DataFrame:
    return np.log(close_df).diff()


def download_fred(series_map: dict, start: str, end: str) -> pd.DataFrame:
    macro = pdr.DataReader(list(series_map.keys()), "fred", start, end)
    return macro.rename(columns=series_map).sort_index()


def transform_macro(macro: pd.DataFrame) -> pd.DataFrame:
    macro = macro.copy()

    # growth-style transforms for slow macro variables
    if "cpi" in macro.columns:
        macro["cpi"] = macro["cpi"].pct_change(12)

    if "indpro" in macro.columns:
        macro["indpro"] = macro["indpro"].pct_change(12)

    # credit spread
    if "dbaa" in macro.columns and "daaa" in macro.columns:
        macro["baa_aaa_spread"] = macro["dbaa"] - macro["daaa"]

    # keep vix in levels
    # keep rates / slope / unemployment in levels

    # optional: drop raw corporate bond yields after building spread
    drop_cols = [c for c in ["dbaa", "daaa"] if c in macro.columns]
    macro = macro.drop(columns=drop_cols, errors="ignore")

    return macro


def write_eval_jsonl_for_split(
    split_name: str,
    split_range: tuple[str, str],
    firm_rets: pd.DataFrame,
    market_ret: pd.Series,
    macro: pd.DataFrame,
    out_path: str,
    min_len: int = 32,
):
    a, b = split_range

    n_seq = 0
    n_tickers = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for t in firm_rets.columns:
            df = pd.DataFrame(index=firm_rets.index)
            df["sequence"] = firm_rets[t]
            df["market_ret_1d"] = market_ret
            for c in macro.columns:
                df[c] = macro[c]

            df = df.loc[a:b].dropna()

            if len(df) < min_len:
                continue

            obj = {
                "ticker": t,
                "segment_id": 0,
                "dates": [d.strftime("%Y-%m-%d") for d in df.index],
                "sequence": df["sequence"].astype(float).tolist(),
                "main_features": df[["sequence", "market_ret_1d"]].astype(float).values.tolist(),
                "macro_features": df[list(macro.columns)].astype(float).values.tolist(),
            }
            f.write(json.dumps(obj) + "\n")
            n_seq += 1
            n_tickers += 1

    print(f"[{split_name}] wrote {n_seq} eval sequences from {n_tickers} tickers -> {out_path}")


def main():
    ensure_dirs()

    tickers = get_sp500_tickers()
    all_tickers = sorted(set(tickers + [MARKET]))

    close = download_close(all_tickers, START, END)
    rets = log_returns(close)

    market_ret = rets[MARKET].rename("market_ret_1d")
    firm_rets = rets.drop(columns=[MARKET], errors="ignore")

    macro = download_fred(FRED_SERIES, START, END)
    macro = transform_macro(macro)

    idx = rets.index
    macro = macro.reindex(idx).ffill().shift(1)

    debug_df = pd.DataFrame(index=idx)
    debug_df["market_ret_1d"] = market_ret
    for c in macro.columns:
        debug_df[c] = macro[c]
    debug_df.to_csv(os.path.join(OUT_ROOT, "metadata", "aligned_market_macro_eval_debug.csv"))

    for split_name, split_range in SPLITS.items():
        out_path = os.path.join(OUT_ROOT, "eval_jsonl_no_fred", f"{split_name}.jsonl")
        write_eval_jsonl_for_split(
            split_name=split_name,
            split_range=split_range,
            firm_rets=firm_rets,
            market_ret=market_ret,
            macro=macro,
            out_path=out_path,
            min_len=32,
        )
    pd.Series(list(macro.columns), name="macro_feature").to_csv(
        os.path.join(OUT_ROOT, "metadata", "macro_feature_order.csv"),
        index=False
    )

    print("Done.")
    print(f"Eval JSONLs are in: {os.path.join(OUT_ROOT, 'eval_jsonl')}")


if __name__ == "__main__":
    main()